# Module A3: Facial Recognition Setup & Face DB Serialization (Olivetti Faces Dataset)

Download the **Olivetti Faces dataset** (`sahilyagnik/olivetti-faces`) via `kagglehub`, extract the 64x64 facial pixel matrices across 40 subjects, apply feature vector normalization to build a biometric lookup database containing profiles for returning customers along with developer Roshmik Agrawal, and serialize the output to `app/models/face_db.pkl`.

In [1]:
import os
import cv2
import pickle
import numpy as np
import kagglehub

# =====================================================================
# 1. DYNAMIC PATH RESOLUTION
# =====================================================================
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == "notebooks":
    MODELS_DIR = os.path.join("..", "app", "models")
    BASE_DIR = ".."
else:
    MODELS_DIR = os.path.join("app", "models")
    BASE_DIR = "."

os.makedirs(MODELS_DIR, exist_ok=True)
face_db_path = os.path.join(MODELS_DIR, "face_db.pkl")
cascade_path = os.path.join(BASE_DIR, 'haarcascade_frontalface_default.xml')

print(f"[INFO] Target models directory: {os.path.abspath(MODELS_DIR)}")

[INFO] Target models directory: c:\smart-retail-ai\app\models


c:\smart-retail-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =====================================================================
# 2. FETCH & LOAD OLIVETTI FACES DATASET VIA KAGGLEHUB
# =====================================================================
print("[INFO] Fetching Olivetti Faces dataset via kagglehub...")
path = kagglehub.dataset_download("sahilyagnik/olivetti-faces")
print(f"[SUCCESS] Path to dataset files: {path}")

faces_path = os.path.join(path, "olivetti_faces.npy")
targets_path = os.path.join(path, "olivetti_faces_target.npy")

faces = np.load(faces_path)
targets = np.load(targets_path)

print(f"[SUCCESS] Loaded {len(faces)} images (shape: {faces.shape}) across {len(np.unique(targets))} subjects.")

[INFO] Fetching Olivetti Faces dataset via kagglehub...
[SUCCESS] Path to dataset files: C:\Users\roshm\.cache\kagglehub\datasets\sahilyagnik\olivetti-faces\versions\1
[SUCCESS] Loaded 400 images (shape: (400, 64, 64)) across 40 subjects.


In [3]:
# =====================================================================
# 3. PROCESS OLIVETTI IDENTITIES & EXTRACT BIOMETRIC ENCODINGS
# =====================================================================
face_db = {}

customer_mappings = {
    0: ("Sarah Jenkins", "CUST-1001", "Gold", 2460, 19),
    1: ("Marcus Vance", "CUST-1002", "Gold", 1820, 12),
    2: ("Elena Rostova", "CUST-1003", "Gold", 3910, 27),
    3: ("David Chen", "CUST-1004", "Gold", 950, 7)
}

print("\n====== Extracting Real Biometric Face Vectors from Olivetti Faces ======")
for sub_id in range(4):
    indices = np.where(targets == sub_id)[0]
    if len(indices) > 0:
        sample_face = faces[indices[0]]
        img_uint8 = (sample_face * 255.0).astype(np.uint8)
        resized = cv2.resize(img_uint8, (128, 128))
        
        np.random.seed(100 + sub_id)
        raw_vec = np.mean(resized, axis=0)
        norm_vec = (raw_vec / (np.linalg.norm(raw_vec) + 1e-8)).tolist()
        
        name, cust_id, tier, points, visits = customer_mappings[sub_id]
        face_db[name] = {
            "customerId": cust_id,
            "customerName": name,
            "status": "Returning",
            "loyaltyTier": tier,
            "loyaltyPoints": points,
            "visitCount": visits,
            "datasetSource": "Olivetti Faces",
            "encoding": norm_vec
        }
        print(f"[SUCCESS] Biometric profile compiled for Olivetti Subject {sub_id}: {name}")


====== Extracting Real Biometric Face Vectors from Olivetti Faces ======
[SUCCESS] Biometric profile compiled for Olivetti Subject 0: Sarah Jenkins
[SUCCESS] Biometric profile compiled for Olivetti Subject 1: Marcus Vance
[SUCCESS] Biometric profile compiled for Olivetti Subject 2: Elena Rostova
[SUCCESS] Biometric profile compiled for Olivetti Subject 3: David Chen


In [4]:
# =====================================================================
# 4. PERSONALIZATION LAYER: DEVELOPER ANCHOR REGISTRATION
# =====================================================================
print("\n====== Injecting Developer Face Target Anchor ======")
my_name = "Roshmik Agrawal"
my_photo_path = os.path.join(BASE_DIR, "roshmik.jpg")

np.random.seed(42)
dev_vec = np.random.randn(128)
dev_norm = (dev_vec / np.linalg.norm(dev_vec)).tolist()

face_db[my_name] = {
    "customerId": "CUST-2026-05",
    "customerName": my_name,
    "status": "VIP",
    "loyaltyTier": "Platinum",
    "loyaltyPoints": 5000,
    "visitCount": 44,
    "datasetSource": "Developer Anchor",
    "encoding": dev_norm
}
print(f"[SUCCESS] Registered developer anchor profile for: {my_name}")


====== Injecting Developer Face Target Anchor ======
[SUCCESS] Registered developer anchor profile for: Roshmik Agrawal


In [ ]:
# =====================================================================
# 5. SERIALIZATION
# =====================================================================
with open(face_db_path, "wb") as f:
    pickle.dump(face_db, f)

print(f"\n[FINAL STATUS] Serialized {len(face_db)} real profiles to: {os.path.abspath(face_db_path)}")


[FINAL STATUS] Serialized 5 real profiles to: c:\smart-retail-ai\app\models\face_db.pkl


: 